# Stages

## Overview

This page covers repeating part of a computation over many values, such as a list of files.

A workflow often has a part that runs once, such as loading a calibration or building a mask, and a part that runs per input: per file, per detector bank, or per chunk of a stream.
The results of the per-input part are then combined, and the rest of the workflow runs once on the combined value.

Sciline expresses this with [Stage](../generated/classes/sciline.Stage.rst), which cuts a flat pipeline into the part that runs once and the part that runs per input.
Accumulators combine the per-input results, and [split](../generated/functions/sciline.split.rst) builds the stages of nested loops, such as files and detector banks.
Nothing is added to the pipeline itself.

## A stage

We start with a pipeline similar to the one in [Getting Started](getting-started.ipynb).
Loading the calibration stands in for an expensive step that does not depend on the file.
The provider appends to a list so we can count how often it runs:

In [ ]:
import math
import operator
from typing import NewType

import sciline

_fake_filesystem = {
    'file102.txt': [1.0, 2.0, float('nan'), 3.0],
    'file103.txt': [1.0, 2.0, 3.0, 4.0],
    'file104.txt': [1.0, 2.0, 3.0, 4.0, 5.0],
    'file105.txt': [1.0, 2.0, 3.0],
    'calib_2025.txt': [0.5],
    'calib_2026.txt': [0.25],
}

# 1. Define domain types

Filename = NewType('Filename', str)
RawData = NewType('RawData', list)
CalibrationFilename = NewType('CalibrationFilename', str)
Calibration = NewType('Calibration', float)
Calibrated = NewType('Calibrated', list)
Total = NewType('Total', float)
ScaleFactor = NewType('ScaleFactor', float)
Result = NewType('Result', float)

# 2. Define providers

calibration_loads = []


def load(filename: Filename) -> RawData:
    """Load the data from a file."""
    return RawData(_fake_filesystem[filename])


def load_calibration(filename: CalibrationFilename) -> Calibration:
    """Load the calibration factor."""
    calibration_loads.append(filename)
    return Calibration(_fake_filesystem[filename][0])


def calibrate(raw: RawData, calibration: Calibration) -> Calibrated:
    """Drop NaNs and apply the calibration."""
    return Calibrated([x * calibration for x in raw if not math.isnan(x)])


def total(data: Calibrated) -> Total:
    """Sum the calibrated data."""
    return Total(sum(data))


def scale(total: Total, factor: ScaleFactor) -> Result:
    """Scale the total."""
    return Result(total * factor)


# 3. Create pipeline

providers = [load, load_calibration, calibrate, total, scale]
params = {CalibrationFilename: 'calib_2025.txt', ScaleFactor: 2.0}
pipeline = sciline.Pipeline(providers, params=params)
pipeline.visualize(Result, graph_attr={'rankdir': 'LR'})

`Filename` has no value.
A [Stage](../generated/classes/sciline.Stage.rst) cuts the pipeline at a set of *inputs*, keys whose values are supplied on each call, and computes the values of a set of *outputs* from them.
Everything the outputs need that does not depend on the inputs is computed once, on first use, and held:

In [ ]:
stage = sciline.Stage(pipeline, outputs=(Result,), inputs=(Filename,))
{
    f: stage.compute({Filename: f})[Result]
    for f in ['file102.txt', 'file103.txt', 'file104.txt']
}

`compute` was called three times, but the calibration was loaded once:

In [ ]:
calibration_loads

The stage tells us where it cut the pipeline.
`visualize` draws the part of the pipeline the stage uses, with the inputs, the held part, and the per-call part marked:

In [ ]:
stage.visualize(graph_attr={'rankdir': 'TD'})

`frontier` and `dynamic` are tuples of keys and `keys` is a frozenset.
`frontier` lists the held keys that the per-call part reads, `dynamic` lists the keys whose values depend on the inputs and are computed on each call, and `keys` contains every key the stage uses, held or per-call:

In [ ]:
stage.frontier

In [ ]:
stage.dynamic

In [ ]:
stage.keys

An input does not have to be a parameter.
If it is an intermediate result, its provider and everything upstream of it are cut off, and the stage computes only what lies downstream.
Here `Total` is supplied, so nothing is loaded and only `ScaleFactor` is held; the cut-off keys are not in `keys` either:

In [ ]:
from_total = sciline.Stage(pipeline, outputs=(Result,), inputs=(Total,))
from_total.frontier, from_total.dynamic, from_total.keys

In [ ]:
from_total.compute({Total: 10.0})

A stage is a snapshot of the pipeline at the time it is built.
Set all parameters, except the inputs, before building it; changing the pipeline afterwards does not affect the stage.

## Warming several stages

Two stages built from the same pipeline may share held work.
Each stage computes its held part on first use, so used independently they would both load the calibration.
[warm](../generated/functions/sciline.warm.rst) computes the held parts of several stages in one run, so shared work is done once:

In [ ]:
calibration_loads.clear()
calibrated = sciline.Stage(pipeline, outputs=(Calibrated,), inputs=(Filename,))
totals = sciline.Stage(pipeline, outputs=(Total,), inputs=(Filename,))
sciline.warm(calibrated, totals)
calibration_loads

Each stage keeps only the values at its own frontier, available from `static()`:

In [ ]:
calibrated.static(), totals.static()

## Combining over files

To combine over files, we need two stages: one from `Filename` to the key at which we combine, the *accumulation key*, and one from that key to the result.
Between them, an accumulator combines the values computed per file.
An accumulator is any object with a `push` method and a `value` property, see [Accumulator](../generated/classes/sciline.Accumulator.rst).

Here we combine `Calibrated` by concatenation.
[Buffered](../generated/classes/sciline.Buffered.rst) makes accumulators that hold every pushed value and apply a function to all of them when the combined value is read:

In [ ]:
def concat(*lists: list) -> list:
    return [x for values in lists for x in values]


def compute(contribute, finalize, accumulators, filenames):
    acc = {key: make() for key, make in accumulators.items()}
    for filename in filenames:
        values = contribute.compute({Filename: filename})
        for key in contribute.dynamic_outputs:
            acc[key].push(values[key])
    return finalize.compute({key: a.value for key, a in acc.items()})


files = ['file102.txt', 'file103.txt', 'file104.txt', 'file105.txt']
contribute = sciline.Stage(pipeline, outputs=(Calibrated,), inputs=(Filename,))
finalize = sciline.Stage(pipeline, outputs=(Result,), inputs=(Calibrated,))
compute(contribute, finalize, {Calibrated: sciline.Buffered(concat)}, files)

`Calibrated` was computed per file, the four lists were concatenated, and `Total` and `Result` were computed once from the concatenated list.
`compute` is the *driver*: the code that calls the stages, here in a loop over files, and pushes their values into accumulators.
It makes new accumulators on each call, so a second call does not add to the values of the first.

`Buffered` holds every pushed value.
For a large dense sum over many files, a running total is much cheaper.
[Reduced](../generated/classes/sciline.Reduced.rst) makes accumulators that apply a binary function to the result so far and each pushed value, and hold only the result.
The function must be associative and must not modify its arguments.
Here we combine at `Total` instead, and sum as we go:

In [ ]:
contribute = sciline.Stage(pipeline, outputs=(Total,), inputs=(Filename,))
finalize = sciline.Stage(pipeline, outputs=(Result,), inputs=(Total,))
compute(contribute, finalize, {Total: sciline.Reduced(operator.add)}, files)

The result is the same, but only one number per file is pushed, and the accumulator holds only the running total.

### Push only what depends on the files

`compute` pushes `dynamic_outputs`, not all outputs of the stage.
An output that does not depend on the inputs, such as `Calibration`, is computed once and returned by every call.
Pushed into a sum, it would be counted once per file:

In [ ]:
with_calibration = sciline.Stage(
    pipeline, outputs=(Total, Calibration), inputs=(Filename,)
)
with_calibration.outputs, with_calibration.dynamic_outputs

The final stage takes only the dynamic outputs as inputs.
It computes `Calibration` from the held part of the pipeline:

In [ ]:
finalize_with_calibration = sciline.Stage(
    pipeline,
    outputs=(Result, Calibration),
    inputs=with_calibration.dynamic_outputs,
)
compute(
    with_calibration,
    finalize_with_calibration,
    {Total: sciline.Reduced(operator.add)},
    files,
)

## The steps separately

The loop in `compute` has three steps: compute a contribution per file, push its values into accumulators, and compute the result from the combined values.
They can also be called separately.
A contribution is a plain dictionary, so the steps can run in separate processes, with the contributions serialized between them:

In [ ]:
def combine(contributions):
    acc = sciline.Reduced(operator.add)()
    for contribution in contributions:
        acc.push(contribution[Total])
    return {Total: acc.value}


contributions = {f: contribute.compute({Filename: f}) for f in files}
finalize.compute(combine(contributions.values()))

Combined values can be pushed again, so combining can also proceed in groups.
A caller may hold the contributions by file, as above, to add or remove files without recomputing the rest.
Here we drop one file:

In [ ]:
del contributions['file105.txt']
finalize.compute(combine(contributions.values()))

## Files in parallel

`contribute.compute` is a plain function of the values of the inputs, so any executor can run it.
Sciline does not own parallelism; whoever iterates over the files chooses how.
We warm the stages first, so that the threads do not each compute the held part:

In [ ]:
from concurrent.futures import ThreadPoolExecutor

sciline.warm(contribute, finalize)
with ThreadPoolExecutor() as executor:
    contributions = executor.map(lambda f: contribute.compute({Filename: f}), files)
    combined = combine(contributions)
finalize.compute(combined)

## Results per file without combining

To compute the value of a key for each file without combining, call a stage in a loop.
The `calibrated` stage from above holds the calibration and computes `Calibrated` per file:

In [ ]:
{f: calibrated.compute({Filename: f})[Calibrated] for f in files}

## Nested loops

Suppose each file holds two detector banks, and reading a file is expensive.
We replace `load` by two providers: one reads the file, one selects a bank from it:

In [ ]:
Bank = NewType('Bank', int)
FileContent = NewType('FileContent', list)
NumValues = NewType('NumValues', int)

file_reads = []


def read_file(filename: Filename) -> FileContent:
    """Read all banks of a file."""
    file_reads.append(filename)
    return FileContent(_fake_filesystem[filename])


def select_bank(content: FileContent, bank: Bank) -> RawData:
    """Even values are in bank 0, odd values in bank 1."""
    return RawData(content[bank::2])


def num_values(content: FileContent) -> NumValues:
    return NumValues(len(content))


banked = pipeline.copy()
banked.insert(read_file)
banked.insert(select_bank)
banked.insert(num_values)

We want `Total` summed over all banks of all files, reading each file once.
Written as plain loops that call the providers by hand, the computation is:

```python
calibration = load_calibration(calibration_filename)  # once
summed = 0.0
for filename in filenames:
    content = read_file(filename)  # once per file
    for b in banks:
        raw = select_bank(content, b)  # once per bank
        summed += total(calibrate(raw, calibration))
result = scale(summed, scale_factor)
```

A single stage from `(Filename, Bank)` to `Total` would do the work of both loops in each call, and read each file once per bank.
Instead, each loop becomes a [Part](../generated/classes/sciline.Part.rst).
Its `inputs` are what the loop iterates over, and its `outputs` are what the loop body needs, here the value pushed into the accumulator.
Its `parent` is the part of the enclosing loop.
It tells `split` that while the bank loop runs, the driver holds what the file part computed for the current file.
The order in which loops are nested is a choice of the driver, not a property of the pipeline, so `split` does not infer it.

[split](../generated/functions/sciline.split.rst) builds a stage per part.
What the bank part reads that depends on the file alone, here `FileContent`, becomes an output of the file stage and an input of the bank stage:

In [ ]:
file = sciline.Part(inputs=(Filename,))
bank = sciline.Part(inputs=(Bank,), outputs=(Total,), parent=file)
per_file, per_bank = sciline.split(banked, file, bank)
finalize_banked = sciline.Stage(banked, outputs=(Result,), inputs=(Total,))
per_file.outputs, per_bank.inputs

[visualize_stages](../generated/functions/sciline.visualize_stages.rst) draws the stages together, with a color for what each stage computes per call.
`FileContent` is computed by the file stage and passed to the bank stage, and `Calibration` is held by the bank stage:

In [ ]:
sciline.visualize_stages(
    per_file, per_bank, finalize_banked, graph_attr={'rankdir': 'TD'}
)

`Calibration` depends on neither loop, so the bank stage holds it, as any stage does.

The driver has the shape of the loops above: the file loop calls the file stage, and the bank loop calls the bank stage with what the file stage returned.
Each stage takes exactly its inputs, so we pick them from what the file stage returned:

In [ ]:
def pick(stage, values):
    return {key: values[key] for key in stage.inputs if key in values}


def compute_banked(filenames, banks):
    total = sciline.Reduced(operator.add)()
    for filename in filenames:
        held = per_file.compute({Filename: filename})
        for b in banks:
            total.push(per_bank.compute({**pick(per_bank, held), Bank: b})[Total])
    return finalize_banked.compute({Total: total.value})


file_reads.clear()
compute_banked(files, banks=[0, 1])

Each file was read once:

In [ ]:
file_reads

### Work after combining the banks

Suppose we want a mean per file: the total of the banks of a file, divided by the number of values in the file.
As plain loops:

```python
means = {}
for filename in filenames:
    content = read_file(filename)
    n = num_values(content)
    summed = 0.0
    for b in banks:
        summed += total(calibrate(select_bank(content, b), calibration))
    means[filename] = mean(summed, n)
```

The division runs once per file, after the bank loop.
It is a third part, whose input is `Total`, the accumulation key of the bank part.
Its parent is the file part: like the bank loop, it runs while the driver holds the values of the current file.
Its stage reads `NumValues` from the file part, like the bank stage reads the file content:

In [ ]:
Mean = NewType('Mean', float)


def mean(total: Total, n: NumValues) -> Mean:
    return Mean(total / n)


banked.insert(mean)
after_banks = sciline.Part(inputs=(Total,), outputs=(Mean,), parent=file)
file_stage, bank_stage, mean_stage = sciline.split(banked, file, bank, after_banks)
file_stage.outputs, mean_stage.inputs

In [ ]:
sciline.visualize_stages(
    file_stage, bank_stage, mean_stage, graph_attr={'rankdir': 'TD'}
)

The driver follows the loops above, with a new accumulator for each file:

In [ ]:
def compute_means(filenames, banks):
    means = {}
    for filename in filenames:
        held = file_stage.compute({Filename: filename})
        total = sciline.Reduced(operator.add)()
        for b in banks:
            values = bank_stage.compute({**pick(bank_stage, held), Bank: b})
            total.push(values[Total])
        values = mean_stage.compute({**pick(mean_stage, held), Total: total.value})
        means[filename] = values[Mean]
    return means


compute_means(files, banks=[0, 1])

`held` is in scope for both the bank loop and the division.
This is what `parent=file` declares for both parts.

### What split refuses

An output declared on a part must depend on the part's inputs.
`NumValues` depends on the file alone, so pushed per bank it would be counted twice.
`split` refuses it and names the part it belongs on:

In [ ]:
try:
    counting = sciline.Part(inputs=(Bank,), outputs=(Total, NumValues), parent=file)
    sciline.split(banked, file, counting)
except ValueError as error:
    print(error)

A value that depends on the inputs of a part can be read only by that part and its descendants.
Without `parent=file`, nothing tells `split` that the driver holds the file content while it loops over the banks, so a bank part cannot read it:

In [ ]:
try:
    sciline.split(banked, file, bank, sciline.Part(inputs=(Bank,), outputs=(Total,)))
except ValueError as error:
    print(error)